# EXP011_FUSION_PILOT_V1
CPU-only bounded synthetic-supervision pilot. No forecasting fits, GPU replay, NASA TEST or final TRAIN calibration. Read PROTOCOL.md first. Run All uploads inputs once and saves every detector fit to Drive. A pass only makes the GCN fusion arm eligible for a separately frozen confirmation.

After a hard runtime interruption, terminate the old runtime before setting RELEASE_STALE_LOCK=True once. Otherwise keep it False. Never run concurrent copies.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import hashlib, json, os, sys, subprocess, zipfile, shutil, datetime
RELEASE_STALE_LOCK = False
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/NASA_GCN_EXP011_FUSION_PILOT_V1')
BASE.mkdir(parents=True,exist_ok=True)
BUNDLE=BASE/'EXP011_FUSION_PILOT_V1_bundle.zip'
INPUT=BASE/'EXP010_MECHANISMS_V1_results.zip'
EXPECTED={BUNDLE:'906a63545e98a0b65cff5011cb8e264aefa67c55fbad1ae907ff71192524ee80',INPUT:'4834bffa6d79c85a6a6b3a6da130eff1fad3782d90e3d3f993cab04f683f8962'}
def sha(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
missing=[p for p,h in EXPECTED.items() if not p.exists()]
if missing:
 print('Upload these files once:',[p.name for p in missing])
 uploaded=files.upload()
 for name,b in uploaded.items():
  h=hashlib.sha256(b).hexdigest();matches=[p for p,v in EXPECTED.items() if v==h]
  if not matches:raise RuntimeError('Unexpected upload: '+name)
  matches[0].write_bytes(b)
for p,h in EXPECTED.items():assert p.exists() and sha(p)==h,str(p)
SRC=Path('/content/exp011_fusion_source')
if SRC.exists():shutil.rmtree(SRC)
SRC.mkdir()
with zipfile.ZipFile(BUNDLE) as z:
 for n in z.namelist():assert not Path(n).is_absolute() and '..' not in Path(n).parts
 z.extractall(SRC)
manifest=json.loads((SRC/'MANIFEST.json').read_text())
for name,h in manifest.items():assert sha(SRC/name)==h,name
print('Input and source verified.')


In [ ]:
ENV=Path('/content/exp011_fusion_env')
if not (ENV/'bin/python').exists():
 subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check','virtualenv'],check=True)
 subprocess.run([sys.executable,'-m','virtualenv',str(ENV)],check=True)
PY=ENV/'bin/python'
subprocess.run([str(PY),'-m','pip','install','--disable-pip-version-check','-r',str(SRC/'requirements.txt')],check=True)
RUNENV=os.environ.copy()
RUNENV.update(OPENBLAS_NUM_THREADS='1',OMP_NUM_THREADS='1',MKL_NUM_THREADS='1',PYTHONHASHSEED='0')
SETUP=BASE/'environment';SETUP.mkdir(exist_ok=True)
stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
(SETUP/('pip_freeze_'+stamp+'.txt')).write_bytes(subprocess.check_output([str(PY),'-m','pip','freeze','--all']))
subprocess.run([str(PY),str(SRC/'src/test_pilot.py')],env=RUNENV,check=True)


In [ ]:
RESULTS=BASE/'results'
cmd=[str(PY),str(SRC/'src/pilot.py'),'--input',str(INPUT),'--output',str(RESULTS)]
if RELEASE_STALE_LOCK:cmd.append('--release-lock')
LOGS=BASE/'logs';LOGS.mkdir(exist_ok=True)
with open(LOGS/('run_'+stamp+'.log'),'a') as log:
 proc=subprocess.Popen(cmd,env=RUNENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
 for line in proc.stdout:
  print(line,end='');log.write(line);log.flush()
 code=proc.wait()
 if code:raise RuntimeError('Run stopped safely. Preserve source/input and completed fits. See Drive log; do not change scientific settings.')
subprocess.run([str(PY),str(SRC/'src/audit_results.py'),'--output',str(RESULTS)],env=RUNENV,check=True)


In [ ]:
EXPORTS=BASE/'exports';EXPORTS.mkdir(exist_ok=True)
FULL=EXPORTS/('EXP011_FUSION_PILOT_V1_results_'+stamp+'.zip')
SUMMARY=EXPORTS/('EXP011_FUSION_PILOT_V1_summary_'+stamp+'.zip')
for destination,full in ((FULL,True),(SUMMARY,False)):
 with zipfile.ZipFile(destination,'w',zipfile.ZIP_DEFLATED) as z:
  for p in sorted(RESULTS.rglob('*')):
   if p.is_file() and (full or 'fits' not in p.relative_to(RESULTS).parts):z.write(p,'results/'+str(p.relative_to(RESULTS)))
  for p in sorted(SETUP.glob('*')):
   if p.is_file():z.write(p,'environment/'+p.name)
  z.write(BUNDLE,'source/'+BUNDLE.name)
  for p in sorted(LOGS.glob('*')):
   if p.is_file():z.write(p,'logs/'+p.name)
(EXPORTS/('checksums_'+stamp+'.json')).write_text(json.dumps({p.name:sha(p) for p in (FULL,SUMMARY)},indent=2))
print((RESULTS/'STUDY_CLOSE.json').read_text())
print('Saved:',FULL,SUMMARY)
files.download(str(SUMMARY))
